# Training Curves for the Best Run of Each Model

This notebook selects the best evaluated run for each architecture using reconstructed-volume validation **mean foreground Dice** from `summary.json`. It then plots the epoch-level training history stored in `metrics.csv`.

For continued training runs, the original and continuation histories are joined and the epoch numbers are adjusted. Loss values should only be compared within a run because the models may use different loss functions.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.precision", 4)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RUNS_ROOT = PROJECT_ROOT / "runs"
PLOT_DIR = PROJECT_ROOT / "docs" / "assets"
SAVE_FIGURES = True
PLOT_EPOCH_LIMIT = 100
if SAVE_FIGURES:
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = {
    "fcn8": "FCN-8",
    "unet2d": "2D U-Net",
    "unet2d_modified": "Modified 2D U-Net",
    "unet3d": "Anisotropic 3D U-Net",
}
MODEL_ORDER = list(MODEL_NAMES)
CLASS_COLUMNS = {
    "RV": "val_dice_class_1",
    "Myocardium": "val_dice_class_2",
    "LV": "val_dice_class_3",
}

print(f"Project root: {PROJECT_ROOT}")

## Select the best reconstructed-volume run per architecture

In [ ]:
def infer_model_key(run_name, metadata):
    model = metadata.get("model")
    if model:
        return model
    name = run_name.lower()
    if "fcn8" in name:
        return "fcn8"
    if "unet3d" in name:
        return "unet3d"
    if "unet2d_modified" in name:
        return "unet2d_modified"
    if "unet2d" in name:
        return "unet2d"
    return "unknown"


evaluated_runs = []
for summary_path in sorted(RUNS_ROOT.glob("*/evaluation_*_val/summary.json")):
    payload = json.loads(summary_path.read_text())
    summary = payload["summary"]
    metadata = payload.get("metadata", {})
    run_dir = summary_path.parents[1]
    model_key = infer_model_key(run_dir.name, metadata)
    if model_key not in MODEL_NAMES or not (run_dir / "metrics.csv").exists():
        continue
    evaluated_runs.append({
        "model_key": model_key,
        "model": MODEL_NAMES[model_key],
        "run": run_dir.name,
        "run_dir": run_dir,
        "mean_volume_dice": summary["mean_dice"],
        "mean_volume_assd_mm": summary["mean_assd_mm"],
        "mean_volume_hd_mm": summary["mean_hd_mm"],
        "checkpoint": metadata.get("checkpoint", ""),
        "postprocessing": metadata.get("postprocessing", "not recorded"),
    })

evaluated_df = pd.DataFrame(evaluated_runs)
best_runs = (
    evaluated_df.sort_values("mean_volume_dice", ascending=False)
    .drop_duplicates("model_key")
    .set_index("model_key")
    .loc[MODEL_ORDER]
    .reset_index()
)
display(best_runs[[
    "model", "run", "mean_volume_dice", "mean_volume_assd_mm",
    "mean_volume_hd_mm", "postprocessing"
]])

## Load and join training histories

If a run was initialized from another run's checkpoint, its parent history is prepended. This is relevant to the selected FCN-8 continuation run.

In [ ]:
def project_path(raw_path):
    if not raw_path:
        return None
    path = Path(str(raw_path).replace("\\", "/"))
    return path if path.is_absolute() else PROJECT_ROOT / path


def load_training_history(run_dir):
    current = pd.read_csv(run_dir / "metrics.csv")
    current["history_source"] = run_dir.name
    current["epoch_global"] = current["epoch"]
    continuation_offset = 0

    config_path = run_dir / "config.json"
    config = json.loads(config_path.read_text()) if config_path.exists() else {}
    weights_path = project_path(config.get("args", {}).get("weights"))
    if weights_path is not None:
        parent_metrics = weights_path.parent / "metrics.csv"
        if parent_metrics.exists() and parent_metrics.parent != run_dir:
            parent = pd.read_csv(parent_metrics)
            parent["history_source"] = parent_metrics.parent.name
            parent["epoch_global"] = parent["epoch"]
            continuation_offset = int(parent["epoch_global"].max())
            current["epoch_global"] = current["epoch"] + continuation_offset
            current = pd.concat([parent, current], ignore_index=True)

    return current.sort_values("epoch_global").reset_index(drop=True), continuation_offset


histories = {}
continuation_offsets = {}
for row in best_runs.itertuples():
    history, offset = load_training_history(Path(row.run_dir))
    histories[row.model_key] = history
    continuation_offsets[row.model_key] = offset
    print(f"{row.model}: {len(history)} epochs plotted from {history['history_source'].nunique()} training stage(s)")

## Per-model training dashboards

The dashed vertical line marks the checkpoint used for reconstructed-volume evaluation. Pixel accuracy is shown for completeness, but it is strongly influenced by the large background class.

In [ ]:
def evaluated_epoch(checkpoint, offset):
    match = re.search(r"best_epoch_(\d+)", str(checkpoint))
    return offset + int(match.group(1)) if match else None


def add_checkpoint_marker(axes, epoch):
    if epoch is None:
        return
    for axis in axes.flat:
        axis.axvline(epoch, color="black", linestyle="--", linewidth=1, alpha=0.65, label="evaluated checkpoint")


def plot_dashboard(model_key, row):
    history = histories[model_key]
    if model_key == "fcn8":
        history = history[history["epoch_global"] <= PLOT_EPOCH_LIMIT]
    checkpoint_epoch = evaluated_epoch(row.checkpoint, continuation_offsets[model_key])
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

    axes[0, 0].plot(history["epoch_global"], history["train_loss"], label="Train")
    axes[0, 0].plot(history["epoch_global"], history["val_loss"], label="Validation")
    axes[0, 0].set(title="Loss", xlabel="Epoch", ylabel="Loss")
    positive_losses = history[["train_loss", "val_loss"]].to_numpy().ravel()
    positive_losses = positive_losses[positive_losses > 0]
    if len(positive_losses) and positive_losses.max() / positive_losses.min() > 20:
        axes[0, 0].set_yscale("log")

    axes[0, 1].plot(history["epoch_global"], history["train_mean_foreground_dice"], label="Train")
    axes[0, 1].plot(history["epoch_global"], history["val_mean_foreground_dice"], label="Validation")
    axes[0, 1].set(title="Mean foreground Dice", xlabel="Epoch", ylabel="Dice", ylim=(0, 1))

    axes[1, 0].plot(history["epoch_global"], history["train_pixel_accuracy"], label="Train")
    axes[1, 0].plot(history["epoch_global"], history["val_pixel_accuracy"], label="Validation")
    axes[1, 0].set(title="Pixel accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))

    for class_name, column in CLASS_COLUMNS.items():
        axes[1, 1].plot(history["epoch_global"], history[column], label=class_name)
    axes[1, 1].set(title="Validation Dice by structure", xlabel="Epoch", ylabel="Dice", ylim=(0, 1))

    if model_key != "fcn8" or checkpoint_epoch is None or checkpoint_epoch <= PLOT_EPOCH_LIMIT:
        add_checkpoint_marker(axes, checkpoint_epoch)
    if model_key == "fcn8":
        for axis in axes.flat:
            axis.set_xlim(1, PLOT_EPOCH_LIMIT)
    for axis in axes.flat:
        handles, labels = axis.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        axis.legend(by_label.values(), by_label.keys(), fontsize=9)

    fig.suptitle(f"{row.model} — {row.run}", fontsize=15)
    if SAVE_FIGURES:
        fig.savefig(PLOT_DIR / f"{model_key}_training_dashboard.png", dpi=180, bbox_inches="tight")
    plt.show()


for row in best_runs.itertuples():
    plot_dashboard(row.model_key, row)

## Cross-model validation Dice

This plot compares learning progress using validation-loader Dice. It is useful for visualizing convergence, but the final model comparison should use the reconstructed-volume metrics shown in the selection table above.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for row in best_runs.itertuples():
    history = histories[row.model_key]
    history = history[history["epoch_global"] <= PLOT_EPOCH_LIMIT]
    ax.plot(history["epoch_global"], history["val_mean_foreground_dice"], label=row.model, linewidth=2)
ax.set(
    title=f"Validation mean foreground Dice during the first {PLOT_EPOCH_LIMIT} epochs",
    xlabel="Epoch", ylabel="Validation Dice", ylim=(0, 1), xlim=(1, PLOT_EPOCH_LIMIT)
)
ax.legend()
if SAVE_FIGURES:
    fig.savefig(PLOT_DIR / "best_models_validation_dice.png", dpi=180, bbox_inches="tight")
plt.show()

## Best logged epoch summary

In [ ]:
summary_rows = []
for row in best_runs.itertuples():
    history = histories[row.model_key]
    best_index = history["val_mean_foreground_dice"].idxmax()
    best_epoch = history.loc[best_index]
    summary_rows.append({
        "Model": row.model,
        "Selected run": row.run,
        "Best logged epoch": int(best_epoch["epoch_global"]),
        "Best loader validation Dice": best_epoch["val_mean_foreground_dice"],
        "Reconstructed-volume Dice": row.mean_volume_dice,
        "Reconstructed-volume ASSD (mm)": row.mean_volume_assd_mm,
        "Reconstructed-volume HD (mm)": row.mean_volume_hd_mm,
    })
display(pd.DataFrame(summary_rows))